<a href="https://colab.research.google.com/github/Mdola73/CodeAlpha_Project_Name/blob/main/Sparse_Dense_Retrieval_In_Class_Activity_9_23.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sparse and Dense Passage Retrieval with Pyserini

A question-answering system normally has two stages:
- **retrieval** that finds passages which may contain the answer, and
- **generator** extracts or writes the answer.

In this lab, we are going to focus on the retrieval. You will compare:
- **Sparse retrieval (Lucene/BM25):** It matches weighted words and is a better match for question-answering that involves names, dates and exact terminology.

- **Dense retrieval (TCT-ColBERT + FAISS):** This maps questions and passages to vectors, thereby allowing semantic matches without identical wording.

Both systems search the same deduplicated subset of 5000 SQuAD examples.

You will be learning to evaluate them with metrics **Recall@1/5/10** and **Mean Reciprocal Rank (MRR)**

### Sparse retrieval

Sparse retrieval represents text through weighted vocabulary terms.

Lucene builds an **inverted index** that maps each term to the passages containing it.

<img src="https://raw.githubusercontent.com/rohinidas18/ta-inclass-activity/main/sparse.png"
     alt="Sparse retrieval pipeline"
     width="100%">

At query time, BM25 rewards matches involving frequent query terms, rare collection terms and appropriate document lengths.

### Dense retrieval

Dense retrieval uses neural encoders to place questions and passages in a shared vector space. Passage vectors are computed in advance and stored in FAISS.

<img src="https://raw.githubusercontent.com/rohinidas18/ta-inclass-activity/main/dense.png"
     alt="Dense retrieval pipeline"
     width="100%">

At query time, the nearest passage vectors are returned according to similarity with the question vector.

### Let's see a worked example

We search four passages:

- **P1:** _Tyrannosaurus rex had powerful jaws and large teeth._
- **P2:** _Triceratops had three large horns on its skull._
- **P3:** _Stegosaurus had rows of bony plates along its back._
- **P4:** _Velociraptor was a small, fast predator._

#### Sparse retrieval

Query: **“Which dinosaur had three large horns?”**

**1. Preprocess the query.** After tokenization and stopword removal, we have: `{dinosaur, three, large, horns}`

**2. Look up terms in the inverted index.** Each term points to passages containing it:

- `three → [P2]`
- `large → [P1, P2]`
- `horns → [P2]`

**3. Form candidates.** The union of the postings lists gives $C=\{P_1,P_2\}$. P2 matches all three informative terms; P1 matches only `large`.

**4. Rank candidates with BM25.**

$$
\mathrm{BM25}(Q,d)=\sum_{t\in Q}\mathrm{IDF}(t)
\frac{f(t,d)(k_1+1)}{f(t,d)+k_1\left(1-b+b\frac{|d|}{\mathrm{avgdl}}\right)}
$$

where:

- $f(t,d)$ is the count of term $t$ in passage $d$;
- $\mathrm{IDF}(t)$ gives more weight to rare terms;
- $|d|/\mathrm{avgdl}$ corrects for passage length;
- $k_1$ controls term-frequency saturation and $b$ controls length normalization.

P2 receives contributions from `three`, `large`, and `horns`, so it ranks above P1.

---

Now say we ask a query: **“Which dinosaur was protected by armor along its body?”**

The intended answer is P3, but its words are `{bony, plates, back}`.

Because `armor` does not literally equal `bony plates`, the important postings may be empty and sparse retrieval may miss P3.

---

#### Dense retrieval

**1. Encode every passage once.** A passage encoder maps each passage to a learned vector. FAISS stores these vectors as the searchable index.

$$
X=\begin{bmatrix}
0.90&0.10&0.40&0.20\\
0.75&0.20&0.30&0.15\\
0.82&0.71&0.15&0.64\\
0.10&0.20&0.05&0.85
\end{bmatrix}
$$

Rows of $X$ correspond to P1–P4.

**2. Encode the question.** A compatible query encoder maps the armor question into the same vector space:

$$
\mathbf q=\begin{bmatrix}0.80&0.69&0.18&0.61\end{bmatrix}
$$

**3. Compare the query with every passage vector.** Using cosine similarity:

$$
\mathrm{sim}(\mathbf q,\mathbf x_i)=
\frac{\mathbf q\cdot\mathbf x_i}{\|\mathbf q\|\,\|\mathbf x_i\|}
$$

For P3, for example:

$$
\mathbf q\cdot\mathbf x_3=1.5624,
\qquad \|\mathbf q\|\approx1.233,
\qquad \|\mathbf x_3\|\approx1.268
$$

$$
\mathrm{sim}(\mathbf q,\mathbf x_3)
=\frac{1.5624}{1.233\times1.268}\approx1.000
$$

**4. Rank by similarity.** The four scores are approximately:

$$
(P_1,P_2,P_3,P_4)=(0.789,0.847,1.000,0.687)
$$

Therefore $P_3>P_2>P_1>P_4$, so FAISS returns P3 first. Dense retrieval succeeds because the encoder can place **armor** and **bony plates** close together despite the vocabulary mismatch.

### Tradeoff

The main trade-off between sparse and dense retrieval is **lexical precision versus semantic matching**.

Sparse retrieval is efficient and interpretable but depends on vocabulary overlap; dense retrieval can connect paraphrases but requires a compatible trained encoder and more computation.

## 1. Setup

Now that you have the background, we implement the two pipelines from the theory section using [**Pyserini**](https://github.com/castorini/pyserini)

| Methodology | What implements it here |
|---|---|
| Sparse retrieval: tokenize, inverted index, BM25 | **Apache Lucene** (via **Anserini**, wrapped by Pyserini’s `LuceneSearcher`) |
| Dense retrieval: encode passages, store vectors, nearest neighbors | A **TCT-ColBERT** encoder + **FAISS**, wrapped by Pyserini’s `FaissSearcher` |
| One Python API for both | Pyserini |

- **Lucene** is the production search library that stores the inverted index.
    - Each term points to a *postings list* of passage IDs, which is exactly the `three → [P2]`, `horns → [P2]` lookup in the dinosaur example.
    - At query time Lucene’s default ranking is **BM25**.

- **Anserini** is an information-retrieval research toolkit built *on* Lucene (indexes, evaluators, standard IR collections).

- **Pyserini** is the Python front end: you call `LuceneSearcher.search(question)` instead of writing Java, but the JVM still does the inverted-index work. That is why this notebook installs **OpenJDK** and sets `JAVA_HOME`.

- **FAISS** stores the passage matrix $X$ from the dense example and returns nearest neighbors of the query vector $\mathbf{q}$. Pyserini still orchestrates encoding and search so both systems look similar in Python.

The zip file _squad_retrieval_artifacts_ that we provide contains the corpus, gold question-to-passage mapping, Lucene index, FAISS index, and a build manifest.

Run the installation once. If Colab requests a restart, restart and continue with the Drive cell. A GPU would help if we encoded passages live, but we provide those vectors for you in the zip file.

In [1]:
# Install a JDK: Pyserini sparse search is backed by Lucene/Anserini, which runs on the JVM.
# Lucene and Anserini run on the **JVM**. Pyserini starts Lucene as a Java process.
# If Java is missing, sparse search fails even when `pip install pyserini` succeeded.

!apt-get update -qq
!apt-get install -y -qq openjdk-21-jdk-headless > /dev/null

# Pyserini = retrieval APIs; faiss-cpu = dense nearest-neighbor search
# transformers = neural encoders.
!pip install -q "pyserini==2.4.0" "faiss-cpu==1.12.0" "transformers>=5,<6"

# Important! Make sure to comment this line after running this cell the first time.
# Your runtime may get disconnected after running this cell, so make sure to
# comment this line and run this cell before proceeding to the following cells.
#!pip install --upgrade --force-reinstall --no-cache-dir pillow

import os
# Lucene subprocesses look up JAVA_HOME. This path is the usual Colab OpenJDK 21 location.
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
print("Setup complete.")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Setup complete.


>> The folder where the zip file is stored: [here](https://drive.google.com/drive/folders/1OnRE-9c9cxpuAjpMf1r5l6JpfSX3kg8p?usp=sharing)  
>> The folder name should be: in_class_activity_9_23  
>> Make sure you move the folder from "Shared With Me" to your "My Drive"

In [7]:
from google.colab import drive
from pathlib import Path
import json, shutil

# Mount Drive so Colab can read the zip you copied into My Drive
drive.mount("/content/drive")

# TODO: Fill out the folder name
ZIP_PATH = Path("/content/drive/MyDrive/in_class_activity_9_23/squad_retrieval_artifacts.zip")

# - `indexes/sparse`: Lucene’s inverted index (term → postings lists), already scored with BM25
# - `indexes/dense`: FAISS’s vector index (passage embeddings)
# - `data/corpus.jsonl` / `data/questions.jsonl`: the passages $d$ and questions $Q$, plus gold IDs

# We ship them as a zip because building a Lucene index and encoding thousands of
# passages is the expensive “index time” half of retrieval.

# Local folder where we will unpack the corpus, gold labels, and prebuilt indexes.
ARTIFACT_ROOT = Path("/content/retrieval_artifacts")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# Optional reference only. It shows how you would feed documents into Pyserini if you were indexing from scratch.
# Lucene does not read a Python list. Anserini’s JsonCollection expects files of objects with `id`
# (stable document ID) and `contents` (the text that will be tokenized into the inverted index).
# Dense encoding typically wants the same fields as **JSONL** (one object per line), which becomes
#  the rows of the FAISS matrix.

# import os
# import json
#
# question = "What is Eric Nyberg's job title?"
# contexts = ["Eric Nyberg is a teacher.", "Nyberg is a musician."]
# corpus = [question] + contexts
#
# # Save corpus data to a JSON file within the folder for sparse retrieval
# corpus_data = [{"id": f"doc{i}", "contents": text} for i, text in enumerate(corpus)]
#
# folder_name = 'corpus_folder'
# os.makedirs(folder_name, exist_ok=True)
#
# corpus_data_file = 'corpus_data.json'
# json_file_path = os.path.join(folder_name, corpus_data_file)
# with open(json_file_path, 'w') as f:
#     json.dump(corpus_data, f)
#
# print(f"Corpus data saved to '{json_file_path}'")
#
# save data for dense retrieval
# with open('corpus_data.jsonl', 'w') as f:
#     for entry in corpus_data:
#         f.write(json.dumps(entry) + '\n')


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# Fail fast if the zip path is wrong (usually the Drive folder was not copied to My Drive)
if not ZIP_PATH.is_file():
    raise FileNotFoundError(f"Artifact ZIP not found: {ZIP_PATH}\nUpdate ZIP_PATH above.")

# Re-unpack from a clean directory
if ARTIFACT_ROOT.exists():
    shutil.rmtree(ARTIFACT_ROOT)
shutil.unpack_archive(str(ZIP_PATH), "/content")

# Confirm the zip really contains the data and both indexes we will search.
required_paths = {
    "manifest": ARTIFACT_ROOT / "manifest.json",
    "corpus": ARTIFACT_ROOT / "data" / "corpus.jsonl",
    "questions": ARTIFACT_ROOT / "data" / "questions.jsonl",
    "sparse index": ARTIFACT_ROOT / "indexes" / "sparse",
    "dense index": ARTIFACT_ROOT / "indexes" / "dense",
}
missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Incomplete artifact ZIP; missing: {missing}")

# The manifest records how the indexes were built (encoder name, collection size, etc.).
# *Query and passage encoders must be a compatible pair (same vector space)
manifest = json.loads(required_paths["manifest"].read_text(encoding="utf-8"))
print(json.dumps(manifest, indent=2))


{
  "artifact_format": 1,
  "dataset": "rajpurkar/squad",
  "dataset_slice": "train[:5000]",
  "passage_count": 820,
  "question_count": 5000,
  "dense_encoder": "castorini/tct_colbert-v2-hnp-msmarco",
  "sparse_index": "Lucene BM25",
  "created_at_utc": "2026-09-20T18:49:43.058958+00:00",
  "build_platform": "Windows-11-10.0.26200-SP0",
  "files": {
    "data/corpus.jsonl": "8e54778d4011ad1e3bfd3e5c1ec303bb79f7b49cb4d0486af0db98c456f7801f",
    "data/questions.jsonl": "e3ce489b8f82ba214565724e459d70f6ebd133daf64d7e34f89a610ea1037e37"
  }
}


## 2. Retrieval intuition

For the question **“What is Eric Nyberg's job title?”**, compare:
1. “Eric Nyberg is a teacher.”
2. “Nyberg performs music in a band.”

**Task**: Now we rank the passages by **lexical overlap** (count of shared alphabetic tokens).

- The teacher sentence shares more surface tokens with the question, so a sparse method prefers it.
- Question 1 asks you for a paraphrase where this overlap test could fail while dense retrieval might succeed.


In [11]:
import re

question = "What is Eric Nyberg's job title?"
toy_passages = ["Eric Nyberg is a teacher.", "Nyberg performs music in a band."]

def lexical_overlap(query, passage):
    # Very rough "sparse" signal: count of shared alphabetic tokens, ignoring BM25 weighting.

    tokens = lambda text: set(re.findall(r"[a-z]+", text.lower()))
    # TODO 2:
    # Tokenize the query and passage.
    query_tokens = tokens(query)
    passage_tokens = tokens(passage)

    # TODO 3
    # Return the number of shared tokens
    return len(query_tokens & passage_tokens)

# Rank passages by overlap
# TODO 4:
# Rank passages from highest to lowest lexical overlap
ranked_passages = sorted(toy_passages, key=lambda p: lexical_overlap(question,p), reverse=True)
for rank, passage in enumerate(ranked_passages, 1):
    print(rank, lexical_overlap(question, passage), passage)


1 3 Eric Nyberg is a teacher.
2 1 Nyberg performs music in a band.


### Question 1. Exact words versus meaning

Why did the overlap rank the first passage highest? Give a reworded question for which exact matching may fail but dense retrieval may succeed.

**Answer:** _Write your response here._

## 3. The SQuAD collection

SQuAD pairs a **question** with a **context passage** that contains the answer. For retrieval we treat those contexts as the document collection $D$ and each question as a query $Q$. The gold document is the context that was paired with that question.

- The 5000 training rows reuse many paragraphs, so the artifact build **deduplicates** passages, assigns stable Lucene/FAISS document IDs and stores `gold_docid` on every question.

These JSONL files are what Anserini would have ingested to build the Lucene index and what the encoder would have embedded into FAISS.

**Next**: We load passages and labeled questions, then print collection size and the first gold pair.

- Pyserini searchers return **document IDs**, not full text.
- The index stores IDs in postings lists or vector slots.
- `passage_by_id` is our lookup so we can read the passage and check whether a hit is gold.

When we later call `searcher.search(question, k=5)`, we are essentially asking: Among unique SQuAD contexts, which five documents does BM25 or dense similarity rank the highest?

In [13]:
def read_jsonl(path):
    # One JSON object per line: passages in corpus.jsonl, labeled questions in questions.jsonl
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle]

corpus = read_jsonl(required_paths["corpus"])
questions = read_jsonl(required_paths["questions"])

# TODO 5:
# Build a dictionary mapping: document ID -> passage text
# Example:
# {
#     "doc123": "some passage...",
#     "doc456": "another passage..."
# }

# Map document id -> passage text so we can print gold and retrieved contexts later
passage_by_id = {__________________}

print(f"Unique passages: {len(corpus):,}")
print(f"Questions: {len(questions):,}")

example = questions[0]
print("Question:", example["question"])
print("Gold passage:", passage_by_id[example["gold_docid"]][:300])


Unique passages: 820
Questions: 5,000
Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Gold passage: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is 


## 4. Sparse retrieval with Lucene

This section is the implementation of the **sparse** pipeline from the theory.

1. **Index time:** Anserini/Lucene tokenized each passage, builds the **inverted index** and stores statistics for BM25 (IDF collection frequencies, document lengths).
2. **Query time:** `LuceneSearcher.search` tokenizes the question, fetches postings and ranks with:

$$
\mathrm{BM25}(Q,d)=\sum_{t\in Q}\mathrm{IDF}(t)
\frac{f(t,d)(k_1+1)}{f(t,d)+k_1\left(1-b+b\frac{|d|}{\mathrm{avgdl}}\right)}
$$

- Pyserini’s `LuceneSearcher` is a Python wrapper around the aforementioned funcationality of Lucene.
- Observer the `docid`, a BM25 **score** and a snippet. The score is a ranking weight and it has a different scale from FAISS similarities.

In [14]:
# How the sparse index would be built from a JsonCollection folder. Anserini commands build the inverted index.
# BM25 search only needs the inverted index and term statistics. We provide you a prebuilt Lucene index in the zip

# import pyserini
# !python -m pyserini.index.lucene \
#   --collection JsonCollection \
#   --input corpus_folder \
#   --index indexes/corpus_folder \
#   --generator DefaultLuceneDocumentGenerator \
#   --threads 1 \
#   --storePositions --storeDocvectors --storeRaw


Open the prebuilt Lucene index with Pyserini and retrieve the **top 5 BM25 passages** for the first SQuAD question.

Gold hits are marked with `<-- GOLD`.

1. Lucene analyzes the question (tokens, often lowercasing / stopwords depending on the analyzer).
2. It unions postings lists for those terms: candidate passages must contain at least one matching term.
3. BM25 ranks the candidates.


In [20]:
from pyserini.search.lucene import LuceneSearcher

# TODO 6:
# Open the prebuilt Lucene sparse index (inverted index over unique SQuAD passages).
# docs: https://github.com/castorini/pyserini/blob/master/docs/usage-search.md
sparse_searcher = LuceneSearcher(str(required_paths["sparse index"]))

# TODO 7:
# Search for the first SQuAD question.
# Retrieve the top 5 passages
sparse_hits = sparse_searcher.search(example["question"], k=5)
# Rank the top 5 passages for the first question; mark the gold document if it appears.

for rank, hit in enumerate(sparse_hits, 1):
    # TODO 8:
    # Mark the result if its document ID matches the gold document ID.
    marker = "<-- GOLD" if hit.docid == example["gold_docid"] else ""
    print(f"{rank:2d}. {hit.docid:20s} BM25={hit.score:8.4f} {marker}")
    print("   ", passage_by_id[hit.docid][:180].replace("\n", " "))


 1. ctx_94a99bff525e95b2 BM25= 17.3225 <-- GOLD
    Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing
 2. ctx_9885c0827ff4772f BM25=  6.5590 
    Because of its Catholic identity, a number of religious buildings stand on campus. The Old College building has become one of two seminaries on campus run by the Congregation of Ho
 3. ctx_72896f9cb6056b37 BM25=  5.4381 
    The University of Notre Dame du Lac (or simply Notre Dame /ˌnoʊtərˈdeɪm/ NOH-tər-DAYM) is a Catholic research university located adjacent to South Bend, Indiana, in the United Stat
 4. ctx_763bfd8595072157 BM25=  4.3237 
    Patricia Ebrey writes that Tibet, like Joseon Korea and other neighboring states to the Ming, settled for its tributary status while there were no troops or governors of Ming China


### Question 2. Interpret sparse results

Did the gold passage appear in the first five? Which words likely influenced the ranking? When would sparse retrieval be preferable?

**Answer:** _Write your response here._

## 5. Dense retrieval with TCT-ColBERT and FAISS

- **Index time:** Each passage has been mapped to a vector $\mathbf{x}_i$ with `castorini/tct_colbert-v2-hnp-msmarco` (TCT-ColBERT distilled from ColBERT on MS MARCO). FAISS stored those vectors.

- **Query time:** Pyserini's `FaissSearcher` encodes the question with a **compatible** query encoder, then FAISS returns the $\mathbf{x}_i$ closest to $\mathbf{q}$.

Pyserini serves as the API: `search` returns the same kind of list as Lucene. Passages with zero shared words can rank first if the encoder placed them near the question.


In [ ]:
# How passage embeddings would be encoded into a FAISS index.

# !python -m pyserini.encode \
#     input --corpus corpus_data.jsonl \
#           --fields text \
#           --delimiter "\n" \
#           --shard-id 0 \
#           --shard-num 1 \
#     output --embeddings path/to/output/dir \
#            --to-faiss \
#     encoder --encoder castorini/tct_colbert-v2-hnp-msmarco \
#             --fields text \
#             --batch 32 \
#             --fp16


`python -m pyserini.encode` is the  Pyserini command that **builds the FAISS index**.

This is index-time dense retrieval: it reads JSONL passages, runs the passage encoder, writes vectors `--to-faiss`.


Now we open the FAISS index with `FaissSearcher(index_dir, encoder_name)`.
Retrieve the **top 5 dense passages** for the same question Lucene just ranked and compare *rankings* with the Lucene cell.

> Remember to avoid comparing raw scores across systems: BM25 and dense similarity are different functions on different representations!

In [ ]:
from pyserini.search.faiss import FaissSearcher

# `manifest["dense_encoder"]` is the query encoder.
# Query encoder must match the passage encoder used when the FAISS index was built
# The name is stored in the manifest so we do not mix DPR with TCT-ColBERT

# TODO 9:
# Open the prebuilt FAISS index.
# The query encoder must be compatible with the passage encoder used to build the index
# docs: https://github.com/castorini/pyserini/blob/master/docs/usage-search.md
dense_searcher = FaissSearcher(
    _____________, manifest[_______________]
)

# TODO 10:
# Retrieve the top 5 passages for the SAME question.
dense_hits = _______________

for rank, hit in enumerate(dense_hits, 1):
  # TODO 11:
  # Mark the gold passage
  marker = ________________ if hit.docid == example["gold_docid"] else ""
  print(f"{rank:2d}. {hit.docid:20s} dense={hit.score:9.4f} {marker}")
  print("   ", passage_by_id[hit.docid][:180].replace("\n", " "))


### Question 3. Compare one ranking

Did sparse and dense return the same first passage? Why are their numerical scores not directly comparable?

**Answer:** _Write your response here._

## 6. Evaluate the questions

A single example is often insufficient to determine a good fit.

We evaluate both Pyserini searchers: `LuceneSearcher` (BM25) and `FaissSearcher` (TCT-ColBERT) on the same questions and the same gold `docid`s.

- **Recall@k:** Fraction of questions whose gold passage appears in the first $k$ hits. This asks whether the inverted-index path or the vector path *found* the gold document at all in a short list a reader could use.

- **MRR (Mean Reciprocal Rank):** Average of $1/\mathrm{rank}$ of gold (0 if absent). This rewards putting gold at rank 1 over rank 9.

- **Runtime:** Lucene BM25 is typically cheaper per query than encoding a question with a transformer and searching FAISS.

In [ ]:
from time import perf_counter

# Here we call `.search` on each Pyserini searcher for many questions and report Recall@1/5/10, MRR@10, and seconds.
# Only the scoring function changes (BM25 on overlapping terms vs similarity in encoder space)

def evaluate_retriever(searcher, examples, ks=(1, 5, 10), limit=None):
    """Score a searcher with Recall@k and MRR.

    Recall@k: fraction of questions whose gold passage is in the top k.

    MRR: mean of 1/rank of the gold passage (0 if it is missing from the list).
    """
    selected = examples if limit is None else examples[:limit]
    hits_at_k = {k: 0 for k in ks}
    reciprocal_rank_sum = 0.0
    start = perf_counter()

    for item in selected:

      # TODO 12:
      # Retrieve enough passages to evaluate
      # the largest requested k.
      hits = ______________

      # TODO 13:
      # Convert the search results into a list of document IDs in ranked order.
      ranked = ______________
      gold = item["gold_docid"]

      # TODO 14:
      # For each k, count whether gold occurs somewhere in the first k results.
      for k in ks:
          hits_at_k[k] += ______________

      # TODO 15:
      # Add the reciprocal rank of the gold document
      # rank 1 -> 1
      # rank 2 -> 1/2
      # rank 5 -> 1/5
      # Remember Python list indices start at 0

      if gold in ranked:
        rank = ___________________
        reciprocal_rank_sum += 1 / ______________________

    count = len(selected)
    return {**{f"Recall@{k}": hits_at_k[k] / count for k in ks},
          "MRR@10": reciprocal_rank_sum / count,
          "questions": count, "seconds": perf_counter() - start}

> **Think of why Recall@10 may exceed Recall@1?**
> A larger $k$ can only include more gold hits! If dense retrieval performs better on Recall@1, that often means more semantic matches.

In [ ]:
# TODO 16:
# Start with 100 questions for a quick experiment.
# Change this to None if you want to evaluate all available questions.
EVAL_LIMIT = ______________

retrievers = [
    ("Sparse (BM25)", sparse_searcher),
    ("Dense (TCT-ColBERT)", dense_searcher)
]

for name, searcher in retrievers:
    print(f"\n{name}")

    for metric, value in evaluate_retriever(searcher, questions, limit=EVAL_LIMIT).items():
        print(f"  {metric:>10}: {value:.4f}" if isinstance(value, float) else f"  {metric:>10}: {value}")


### Question 4. Evaluate and explain

Record both systems' Recall@1/5/10, MRR@10, and runtime. Which performed better? Why can Recall@10 exceed Recall@1?

**Answer:** _Write your results and explanation here._

### Question 5. Error analysis

Change the example index in a new code cell and find a question for which the systems rank the gold passage differently. Explain likely lexical or semantic reasons. Is a high-ranked non-gold passage necessarily irrelevant?

**Answer:** _Write your response here._

### Question 6. Passage versus sentence indexing

If contexts were split into sentences, how would the gold-document definition change? Give one benefit and one cost for sparse and dense retrieval.

**Answer:** _Write your response here._

## 7. Takeaways and further reading

_Jot down your takeaways here!_

Feel free to checkout these papers for more context

- **Robertson and Zaragoza (2009), “The Probabilistic Relevance Framework: BM25 and Beyond.”** Definitive account of the probabilistic relevance framework and BM25 used by modern sparse search systems (including Lucene). [DOI](https://doi.org/10.1561/1500000019)

- **Karpukhin et al. (2020), “Dense Passage Retrieval for Open-Domain Question Answering.”** Popularized dual-encoder dense passage retrieval for question answering. [ACL Anthology](https://aclanthology.org/2020.emnlp-main.550/)

- **Lin, Yang, and Lin (2021), “In-Batch Negatives for Knowledge Distillation with Tightly-Coupled Teachers for Dense Retrieval.”** Introduced the training approach behind TCT-ColBERT, the encoder used in this lab. [arXiv](https://arxiv.org/abs/2106.03318)

- **Pyserini / Anserini:** [Pyserini documentation](https://github.com/castorini/pyserini) — Python IR toolkit wrapping Lucene/Anserini and FAISS, which is what you called in this lab.
